In [ ]:

!pip install torch torchvision onnx onnxruntime


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 15.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.8/27.8 MB 52.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [onnx]


In [ ]:

import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import onnx
import os

print("Libraries imported successfully!")
print("PyTorch Version:", torch.__version__)


Libraries imported successfully!
PyTorch Version: 2.14.1+cpu


In [ ]:

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)


Device: cpu


In [ ]:

class CNNModel(nn.Module):
    def __init__(self):
        super(CNNModel, self).__init__()

        self.conv1 = nn.Conv2d(3, 16, 3, padding=1)
        self.relu = nn.ReLU()
        self.pool = nn.MaxPool2d(2, 2)

        self.conv2 = nn.Conv2d(16, 32, 3, padding=1)

        self.fc1 = nn.Linear(32 * 8 * 8, 64)
        self.fc2 = nn.Linear(64, 10)

    def forward(self, x):
        x = self.pool(self.relu(self.conv1(x)))
        x = self.pool(self.relu(self.conv2(x)))

        x = x.view(x.size(0), -1)

        x = self.relu(self.fc1(x))
        x = self.fc2(x)

        return x

model = CNNModel().to(device)

print(model)


CNNModel(
  (conv1): Conv2d(3, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (relu): ReLU()
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv2): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (fc1): Linear(in_features=2048, out_features=64, bias=True)
  (fc2): Linear(in_features=64, out_features=10, bias=True)
)


In [ ]:

transform = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.ToTensor()
])

train_dataset = torchvision.datasets.CIFAR10(
    root="./data",
    train=True,
    download=True,
    transform=transform
)

train_loader = torch.utils.data.DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True
)

print("Dataset loaded successfully!")
print("Total training images:", len(train_dataset))


Dataset loaded successfully!
Total training images: 50000


In [ ]:

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=0.001
)

print("Loss function and optimizer created!")


Loss function and optimizer created!


In [ ]:

epochs = 2

model.train()

for epoch in range(epochs):
    running_loss = 0.0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    print(
        f"Epoch [{epoch+1}/{epochs}], "
        f"Loss: {running_loss/len(train_loader):.4f}"
    )

print("Model training completed!")


Epoch [1/2], Loss: 1.6112
Epoch [2/2], Loss: 1.2796
Model training completed!


In [ ]:

torch.save({
    "model_state_dict": model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    "epochs": epochs
}, "trained_model.pth")

print("Trained checkpoint saved successfully!")


Trained checkpoint saved successfully!


In [ ]:

checkpoint = torch.load(
    "trained_model.pth",
    map_location=device
)

model.load_state_dict(checkpoint["model_state_dict"])

model.eval()

print("Trained checkpoint loaded successfully!")


Trained checkpoint loaded successfully!


In [ ]:

# Load the saved checkpoint safely

checkpoint = torch.load(
    "trained_model.pth",
    map_location=device,
    weights_only=True
)

# Load trained weights into the model
model.load_state_dict(checkpoint["model_state_dict"])

# Set model to evaluation mode
model.eval()

print("Checkpoint loaded successfully!")
print("Model is ready for inference!")


Checkpoint loaded successfully!
Model is ready for inference!


In [ ]:

import torch

# Move model to CPU
model = model.to("cpu")
model.eval()

# Create sample input
dummy_input = torch.randn(1, 3, 32, 32)

print("Model prepared successfully!")
print("Input shape:", dummy_input.shape)


Model prepared successfully!
Input shape: torch.Size([1, 3, 32, 32])


In [ ]:

torch.onnx.export(
    model,
    dummy_input,
    "model.onnx",
    input_names=["input"],
    output_names=["output"],
    opset_version=17,
    dynamo=False
)

print("ONNX model exported successfully!")


ONNX model exported successfully!


/tmp/ipykernel_988/3435713738.py:1: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(


In [ ]:

import onnx

# Load ONNX model
onnx_model = onnx.load("model.onnx")

# Validate model
onnx.checker.check_model(onnx_model)

print("ONNX model validation successful!")
print("Input:", onnx_model.graph.input[0].name)
print("Output:", onnx_model.graph.output[0].name)


ONNX model validation successful!
Input: input
Output: output


In [ ]:

import onnxruntime as ort
import numpy as np

# Create ONNX inference session
session = ort.InferenceSession(
    "model.onnx",
    providers=["CPUExecutionProvider"]
)

# Create sample input
input_data = np.random.randn(
    1, 3, 32, 32
).astype(np.float32)

# Perform prediction
output = session.run(
    ["output"],
    {"input": input_data}
)

predicted_class = np.argmax(output[0], axis=1)

print("ONNX inference successful!")
print("Output shape:", output[0].shape)
print("Predicted class:", predicted_class[0])


ONNX inference successful!
Output shape: (1, 10)
Predicted class: 9


In [ ]:

try:
    import tensorrt as trt

    logger = trt.Logger(trt.Logger.WARNING)

    builder = trt.Builder(logger)

    network = builder.create_network(
        1 << int(
            trt.NetworkDefinitionCreationFlag.EXPLICIT_BATCH
        )
    )

    parser = trt.OnnxParser(network, logger)

    with open("model.onnx", "rb") as f:
        success = parser.parse(f.read())

    if not success:
        for i in range(parser.num_errors):
            print(parser.get_error(i))
        raise RuntimeError("ONNX parsing failed")

    config = builder.create_builder_config()

    config.set_memory_pool_limit(
        trt.MemoryPoolType.WORKSPACE,
        1 << 30
    )

    engine = builder.build_serialized_network(
        network, config
    )

    if engine is None:
        raise RuntimeError("Engine creation failed")

    with open("model.engine", "wb") as f:
        f.write(engine)

    print("TensorRT engine created successfully!")

except ImportError:
    print("TensorRT is not installed.")
    print("Run this step in a compatible NVIDIA environment.")


TensorRT is not installed.
Run this step in a compatible NVIDIA environment.


In [ ]:

import os

files = [
    "trained_model.pth",
    "model.onnx",
    "model.engine"
]

print("MODEL ARTEFACT VERIFICATION")

for file in files:
    if os.path.exists(file):
        print(file, " - Available")
    else:
        print(file, " - Not Available")


MODEL ARTEFACT VERIFICATION
trained_model.pth  - Not Available
model.onnx  - Not Available
model.engine  - Not Available
